# P7: DICOM source-priority test (150 cases)

In [ ]:

import os, subprocess
_q = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                    capture_output=True, text=True)
_sm = _q.stdout.strip().split(".")
SM = (int(_sm[0]), int(_sm[1])) if len(_sm) == 2 and _sm[0].strip().isdigit() else (9, 0)
if SM < (7, 0):
    subprocess.run(["pip", "install", "-q", "torch==2.3.1+cu118", "torchvision==0.18.1+cu118",
                    "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
os.system("pip install -q pydicom albumentations pretrainedmodels efficientnet_pytorch tqdm munch scikit-learn")
os.system("pip install -q --no-deps segmentation-models-pytorch")
import glob, json
import numpy as np, pandas as pd, pydicom
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from typing import List, Dict, Tuple, Optional, Any
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
print("smp:", smp.__version__, "torch:", torch.__version__)
device = torch.device("cuda")

# 1. per-source pixel comparison for 3 test ids
splits = pd.read_csv(glob.glob("/kaggle/input/**/patient_splits.csv", recursive=True)[0])
te = splits[splits["Fold"].isin(["test", "Test"])].reset_index(drop=True)
for iid in te["ImageId"].iloc[[0, 1, 2]]:
    hits = glob.glob(f"/kaggle/input/**/{iid}.dcm", recursive=True)
    print("ID", iid[:20], "nhits", len(hits))
    for h in hits:
        d = pydicom.dcmread(h); a = d.pixel_array
        print("  ", h.split("/kaggle/input/")[1][:60], a.shape, a.dtype, int(a.min()), int(a.max()),
              getattr(d, "PhotometricInterpretation", "?"))

# 2. original priority indexer
def index_orig():
    m = {}
    for sp in ["/kaggle/input/siim-acr-pneumothorax-segmentation-data",
               "/kaggle/input/siim-acr-pneumothorax-segmentation", "/kaggle/input"]:
        if os.path.exists(sp):
            for root, _, files in os.walk(sp):
                for f in files:
                    if f.endswith(".dcm"):
                        m.setdefault(f[:-4], os.path.join(root, f))
    return m
dmap = index_orig()
print("indexed:", len(dmap), "sample:", list(dmap.values())[0].split("/kaggle/input/")[1][:70])


In [ ]:
# 3. SIIM-ACR RLE Decoder with Multi-Mask Logical OR Aggregation

def rle_decode(rle_str: Any, shape: tuple = (1024, 1024)) -> np.ndarray:
    """Decode SIIM-ACR Run-Length Encoded string into binary mask (Fortran order)."""
    if rle_str is None or (isinstance(rle_str, float) and np.isnan(rle_str)):
        return np.zeros(shape, dtype=np.uint8)

    rle_str = str(rle_str).strip()
    if rle_str == "" or rle_str == "-1":
        return np.zeros(shape, dtype=np.uint8)

    s = rle_str.split()
    starts = np.asarray([int(float(x)) for x in s[0::2]], dtype=int) - 1
    lengths = np.asarray([int(float(x)) for x in s[1::2]], dtype=int)
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order="F")

def aggregate_rle_list(rle_list: List[str], shape: tuple = (1024, 1024)) -> np.ndarray:
    """Combine multiple RLE instances for an image using logical OR."""
    composite = np.zeros(shape, dtype=np.uint8)
    for rle in rle_list:
        mask = rle_decode(rle, shape=shape)
        composite = np.bitwise_or(composite, mask)
    return composite

# Verify RLE decoder on sample
sample_rles = ast.literal_eval(df_splits.iloc[0]["EncodedPixelsList"])
sample_mask = aggregate_rle_list(sample_rles)
print(f"Verification: sample mask shape={sample_mask.shape}, sum={np.sum(sample_mask)}")


In [ ]:
# 4. PyTorch Dataset and Radiologically Sound Augmentations

class SIIMPneumothoraxDataset(Dataset):
    """Dataset for SIIM-ACR pneumothorax radiographs with dense masks."""

    def __init__(self, df: pd.DataFrame, image_size: int = 512, transforms: Optional[Any] = None):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.transforms = transforms

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        dcm_path = row["dcm_path"]
        
        # Read DICOM
        try:
            dcm = pydicom.dcmread(dcm_path)
            img = dcm.pixel_array.astype(np.float32)
            if hasattr(dcm, "PhotometricInterpretation") and dcm.PhotometricInterpretation == "MONOCHROME1":
                img = np.amax(img) - img
        except Exception:
            img = np.zeros((1024, 1024), dtype=np.float32)

        # Normalize to uint8 [0, 255]
        img_min, img_max = img.min(), img.max()
        if img_max > img_min:
            img = ((img - img_min) / (img_max - img_min) * 255.0).astype(np.uint8)
        else:
            img = np.zeros_like(img, dtype=np.uint8)

        # Grayscale to 3-channel
        img_3c = np.repeat(np.expand_dims(img, axis=-1), 3, axis=-1)

        # Decode ground-truth mask
        rle_raw = row["EncodedPixelsList"]
        if isinstance(rle_raw, str):
            try:
                rle_list = ast.literal_eval(rle_raw)
            except Exception:
                rle_list = [rle_raw]
        else:
            rle_list = [str(rle_raw)]

        mask = aggregate_rle_list(rle_list, shape=img.shape)

        # Apply Albumentations transforms
        if self.transforms is not None:
            augmented = self.transforms(image=img_3c, mask=mask)
            image_tensor = augmented["image"]
            mask_tensor = augmented["mask"].unsqueeze(0).float()
        else:
            image_tensor = torch.from_numpy(img_3c).permute(2, 0, 1).float() / 255.0
            mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "image_id": row["ImageId"],
            "patient_id": str(row["PatientID"]),
            "view_position": str(row["ViewPosition"]),
            "has_pneumothorax": int(row["HasPneumothorax"])
        }

# Data augmentations: HorizontalFlip, ShiftScaleRotate, RandomBrightnessContrast
# Strictly NO VerticalFlip (violates anatomical cephalocaudal orientation)
train_transforms = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-10, 10), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Partition DataFrames
# Train: Folds 1, 2, 3, 4 | Val: Fold 0 | Test Holdout: Fold 'test'
train_df = df_splits[df_splits["Fold"].isin(["1", "2", "3", "4", 1, 2, 3, 4])].reset_index(drop=True)
val_df = df_splits[df_splits["Fold"].isin(["0", 0])].reset_index(drop=True)
test_df = df_splits[df_splits["Fold"].isin(["test", "Test"])].reset_index(drop=True)

print(f"Train set: {len(train_df)} images ({train_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Validation set: {len(val_df)} images ({val_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Test Holdout set: {len(test_df)} images ({test_df['HasPneumothorax'].mean()*100:.1f}% positive)")

# Verify zero patient leakage
train_val_patients = set(train_df["PatientID"]).union(set(val_df["PatientID"]))
test_patients = set(test_df["PatientID"])
overlap = train_val_patients.intersection(test_patients)
assert len(overlap) == 0, f"DATA LEAKAGE DETECTED: {len(overlap)} overlapping patients!"
print(f"LEAKAGE PROOF VERIFIED: Exactly {len(overlap)} overlapping patients between train/val and test.")


In [ ]:
# 5. Architecture: ResNet34 U-Net with Spatial Dropout and Combined Loss
import segmentation_models_pytorch as smp

class PneumothoraxUNet(nn.Module):
    """ResNet34 U-Net supporting deterministic inference and MC Dropout."""

    def __init__(self, dropout_rate: float = 0.2, pretrained: bool = True):
        super().__init__()
        self.dropout_rate = dropout_rate
        weights = "imagenet" if pretrained else None
        
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            decoder_channels=(256, 128, 64, 32, 16)
        )
        
        # Inject SpatialDropout2d into decoder blocks for MC Dropout
        if dropout_rate > 0.0:
            for idx in range(len(self.model.decoder.blocks)):
                self.model.decoder.blocks[idx].add_module(
                    "spatial_dropout", nn.Dropout2d(p=dropout_rate)
                )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    @torch.no_grad()
    def predict_deterministic(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Deterministic forward pass (eval mode)."""
        self.eval()
        logits = self.forward(x)
        prob = torch.sigmoid(logits)
        eps = 1e-7
        p_clamped = torch.clamp(prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        return {"prob": prob, "entropy": entropy}

    @torch.no_grad()
    def predict_mc_dropout(self, x: torch.Tensor, num_samples: int = 20) -> Dict[str, torch.Tensor]:
        """Monte Carlo Dropout inference with T stochastic passes."""
        self.train() # Activates Spatial Dropout during inference
        samples = []
        for _ in range(num_samples):
            logits = self.forward(x)
            samples.append(torch.sigmoid(logits))
        
        stacked = torch.stack(samples, dim=0) # (T, B, 1, H, W)
        mean_prob = torch.mean(stacked, dim=0)
        variance = torch.var(stacked, dim=0, unbiased=True)
        
        eps = 1e-7
        p_clamped = torch.clamp(mean_prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        
        return {
            "mean": mean_prob,
            "variance": variance,
            "entropy": entropy
        }

# Combined Loss: 0.5 * BCE + 0.5 * SoftDice
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        cardinality = (probs.pow(2) + targets.pow(2)).sum()
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice

class CombinedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice_loss = SoftDiceLoss(smooth=1.0)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_bce = F.binary_cross_entropy_with_logits(logits, targets)
        loss_dice = self.dice_loss(logits, targets)
        return self.bce_weight * loss_bce + self.dice_weight * loss_dice

criterion = CombinedBCEDiceLoss()
print("Model architecture and CombinedBCEDiceLoss verified.")


In [ ]:
# 7. Evaluation Metrics: Disaggregated Dice, ESCE, AUROC-ED, and AURC

def compute_dice_coefficient(y_true: np.ndarray, y_pred: np.ndarray, empty_score: float = 1.0) -> float:
    y_true_sum = np.sum(y_true)
    y_pred_sum = np.sum(y_pred)
    if y_true_sum == 0 and y_pred_sum == 0:
        return empty_score
    if y_true_sum == 0 or y_pred_sum == 0:
        return 0.0
    intersection = np.sum(y_true * y_pred)
    return float(2.0 * intersection / (y_true_sum + y_pred_sum))

def compute_segmentation_metrics(y_true: np.ndarray, y_pred_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_pred_prob >= threshold).astype(np.uint8)
    y_true = (y_true > 0).astype(np.uint8)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    is_positive = float(np.sum(y_true) > 0)
    dice = compute_dice_coefficient(y_true, y_pred)
    iou = float(tp / (tp + fp + fn)) if (tp + fp + fn) > 0 else (1.0 if is_positive == 0 else 0.0)
    sensitivity = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 1.0
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else (1.0 if np.sum(y_pred) == 0 else 0.0)

    return {
        "dice": dice,
        "iou": iou,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "is_positive": is_positive,
    }

def compute_esce(probs: np.ndarray, targets: np.ndarray, n_bins: int = 10):
    probs_flat = probs.flatten()
    targets_flat = targets.flatten()
    total = len(probs_flat)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_confs = np.zeros(n_bins)
    bin_accs = np.zeros(n_bins)
    bin_counts = np.zeros(n_bins)
    esce = 0.0

    for i in range(n_bins):
        low, high = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (probs_flat > low) & (probs_flat <= high) if i > 0 else (probs_flat >= low) & (probs_flat <= high)
        count = np.sum(mask)
        bin_counts[i] = count
        if count > 0:
            conf = np.mean(probs_flat[mask])
            acc = np.mean(targets_flat[mask])
            bin_confs[i] = conf
            bin_accs[i] = acc
            esce += (count / total) * np.abs(acc - conf)

    return float(esce), bin_confs, bin_accs, bin_counts

def compute_brier_score(probs: np.ndarray, targets: np.ndarray) -> float:
    return float(np.mean((probs - targets) ** 2))

def compute_error_detection_auroc(uncertainty_map: np.ndarray, probs: np.ndarray, targets: np.ndarray, threshold: float = 0.5) -> float:
    preds = (probs >= threshold).astype(np.uint8)
    error = np.abs(targets - preds).flatten()
    uncertainty = uncertainty_map.flatten()
    if np.all(error == 0) or np.all(error == 1):
        return 0.5
    if len(error) > 100_000:
        idx = np.random.choice(len(error), size=100_000, replace=False)
        error = error[idx]
        uncertainty = uncertainty[idx]
    try:
        return float(roc_auc_score(error, uncertainty))
    except Exception:
        return 0.5

def aggregate_case_uncertainty(uncertainty_map: np.ndarray, k: int = 500) -> float:
    flat = uncertainty_map.flatten()
    k = min(k, len(flat))
    top_k_vals = np.partition(flat, -k)[-k:]
    return float(np.mean(top_k_vals))

def compute_risk_coverage_curve(case_uncertainties: np.ndarray, case_risks: np.ndarray, steps: int = 50, min_cov: float = 0.20):
    n = len(case_uncertainties)
    sorted_idx = np.argsort(case_uncertainties)
    coverages = np.linspace(min_cov, 1.0, steps)
    risks = np.zeros(steps)
    for i, cov in enumerate(coverages):
        retain_n = max(1, int(round(cov * n)))
        retained = sorted_idx[:retain_n]
        risks[i] = float(np.mean(case_risks[retained]))
    return coverages, risks

def compute_aurc(coverages: np.ndarray, risks: np.ndarray) -> float:
    cov_norm = (coverages - coverages[0]) / (coverages[-1] - coverages[0])
    try:
        from scipy.integrate import trapezoid
        return float(trapezoid(risks, cov_norm))
    except ImportError:
        trapz_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))
        return float(trapz_fn(risks, cov_norm))


In [ ]:

# 150-case det inference with ORIGINAL priority mapping
import torch.nn.functional as F
te150 = te.iloc[:150].copy()
te150["dcm_path"] = te150["ImageId"].map(dmap)
print("mapped:", int(te150["dcm_path"].notna().sum()))
ld = DataLoader(SIIMPneumothoraxDataset(te150, image_size=512, transforms=val_transforms),
                batch_size=1, shuffle=False, num_workers=2)
W = os.path.dirname(glob.glob("/kaggle/input/**/model_seed42.pt", recursive=True)[0]) + "/"
net = PneumothoraxUNet(dropout_rate=0.2, pretrained=False).to(device)
net.load_state_dict(torch.load(W + "model_seed42.pt", map_location=device)); net.eval()
ds = []
with torch.no_grad():
    for b in ld:
        p = torch.sigmoid(net(b["image"].to(device))).cpu().numpy()[0, 0]
        m = b["mask"].numpy()[0, 0]
        ds.append(compute_segmentation_metrics(m, p)["dice"])
print("P7 det_dice@150 =", round(float(np.mean(ds)), 4), "(orig first-150 = 0.191)")
